# Task 3 – Feature Engineering Lead
### OULAD Group Project: Fail vs Pass Prediction Over Time (Weeks 2 / 4 / 6 / 8)

**Author:** Surisha  
**Depends on:** Task 1 (repo + raw CSVs in `data/raw/`), Task 2 (confirm file names + join keys)  
**Outputs:**
- `data/processed/week2/4/6/8_features.csv` — consumed by Tasks 4 and 5
- `data/processed/features_table.csv` — fills Table 2.3 in the PDF submission

| File | Cutoff | Meaning |
|------|--------|---------|
| `week2_features.csv` | date ≤ 13 | First 2 weeks of course |
| `week4_features.csv` | date ≤ 27 | First 4 weeks of course |
| `week6_features.csv` | date ≤ 41 | First 6 weeks of course |
| `week8_features.csv` | date ≤ 55 | First 8 weeks of course |

## 1. Imports & setup

In [1]:
import os
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()
if (BASE_DIR / 'data' / 'interim').exists():
    REPO_DIR = BASE_DIR
else:
    REPO_DIR = BASE_DIR / 'ELEN4025_Group_Assignment_gift_full'
RAW_DIR = REPO_DIR / 'data' / 'raw'
INTERIM_DIR = REPO_DIR / 'data' / 'interim'
OUT_DIR = REPO_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CUTOFFS = {'week2': 13, 'week4': 27, 'week6': 41, 'week8': 55}
EXPECTED_INTERIM_ROWS = {
    'week2': 1_757_383,
    'week4': 2_801_311,
    'week6': 3_562_771,
    'week8': 4_302_651,
}

LABEL_MAP = {'Pass': 1, 'Distinction': 1, 'Fail': 0, 'Withdrawn': 0}

CAT_COLS = ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']
NUM_COLS = ['num_of_prev_attempts', 'studied_credits']

# High-frequency VLE activity types selected for per-type feature columns.
# Low-frequency types (dataplus, url, page, scorm, folder, etc.) excluded to keep
# the feature table manageable. Final list validated against actual Task 2 handoff data in cell 4.
VLE_ACT_TYPES = ['oucontent', 'quiz', 'resource', 'homepage', 'subpage',
                  'glossary', 'oucollaborate', 'forumng']


def ensure_raw_files(file_names):
    """Extract required raw OULAD files from the cloned repo zip when missing."""
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    missing = [name for name in file_names if not (RAW_DIR / name).exists()]
    if not missing:
        return

    zip_path = REPO_DIR / 'anonymisedData.zip'
    if not zip_path.exists():
        raise FileNotFoundError(f'Missing {zip_path}; cannot extract raw OULAD files.')

    with zipfile.ZipFile(zip_path) as zf:
        members = set(zf.namelist())
        for name in missing:
            if name not in members:
                raise FileNotFoundError(f'{name} not found in {zip_path.name}')
            zf.extract(name, RAW_DIR)
    print(f'Extracted raw files: {missing}')


def count_outliers_iqr(series: pd.Series) -> int:
    """Count values outside 1.5xIQR fence (Tukey method)."""
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return 0
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


print(f'Using cloned repo: {REPO_DIR}')
print('Setup complete.')



Using cloned repo: /home/njabulo/Documents/Wits/Final year/Semester_1/Machine_Learning/2026/Projects/Group/ELEN4025_Group_Assignment
Setup complete.


## 2. Load raw OULAD tables
`sum_click` uses `int32` to avoid silent wrap-around.  
`EARLIEST_DATE` is derived from the actual data — never hardcoded.

In [2]:
print('Loading Task 2 handoff files from the cloned gift-task2-cutoffs branch ...')

ensure_raw_files(['studentInfo.csv'])
student_info_raw = pd.read_csv(RAW_DIR / 'studentInfo.csv')

vle_dtypes = {
    'code_module': 'category',
    'code_presentation': 'category',
    'id_student': 'int32',
    'id_site': 'int32',
    'date': 'int16',
    'sum_click': 'int32',
    'activity_type': 'category',
}

student_vle_by_week = {}
for name in CUTOFFS:
    path = INTERIM_DIR / f'studentVle_{name}.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing Task 2 handoff file: {path}')

    df = pd.read_csv(path, dtype=vle_dtypes)
    student_vle_by_week[name] = df

    expected_rows = EXPECTED_INTERIM_ROWS[name]
    status = 'OK' if len(df) == expected_rows else f'EXPECTED {expected_rows:,}'
    print(f'  {path.name}: {df.shape} [{status}]')

EARLIEST_DATE = int(min(df['date'].min() for df in student_vle_by_week.values()))
actual_activity_types = sorted(
    set().union(*(set(df['activity_type'].dropna().astype(str).unique()) for df in student_vle_by_week.values()))
)

print(f'  studentInfo : {student_info_raw.shape}')
print(f'  Earliest VLE date in Task 2 handoff: {EARLIEST_DATE}')
print(f'  Activity types in handoff files: {actual_activity_types}')


Loading Task 2 handoff files from the cloned gift-task2-cutoffs branch ...
Extracted raw files: ['studentInfo.csv']


  studentVle_week2.csv: (1757383, 7) [OK]
  studentVle_week4.csv: (2801311, 7) [OK]
  studentVle_week6.csv: (3562771, 7) [OK]
  studentVle_week8.csv: (4302651, 7) [OK]
  studentInfo : (32593, 12)
  Earliest VLE date in Task 2 handoff: -25
  Activity types in handoff files: ['dataplus', 'dualpane', 'externalquiz', 'forumng', 'glossary', 'homepage', 'htmlactivity', 'oucollaborate', 'oucontent', 'ouelluminate', 'ouwiki', 'page', 'questionnaire', 'quiz', 'repeatactivity', 'resource', 'sharedsubpage', 'subpage', 'url']


## 3. Prepare demographics table
`student_info_raw` is never modified — re-running is always safe.

In [3]:
demo_cols = (
    ['id_student', 'code_module', 'code_presentation']
    + CAT_COLS + NUM_COLS + ['final_result']
)

student_info = student_info_raw[demo_cols].copy()
student_info['label'] = student_info['final_result'].map(LABEL_MAP)
student_info = student_info.dropna(subset=['label'])
student_info['label'] = student_info['label'].astype(int)
student_info = student_info.drop(columns=['final_result'])

print(f'studentInfo after label mapping: {student_info.shape}')
print(student_info['label'].value_counts().rename(
    {0: 'Unfavourable (0)', 1: 'Favourable (1)'}))

studentInfo after label mapping: (32593, 12)
label
Unfavourable (0)    17208
Favourable (1)      15385
Name: count, dtype: int64


## 4. Attach activity type and validate VLE_ACT_TYPES
**Re-run safe:** guard skips merge if column already exists.  
`VLE_ACT_TYPES` is validated against the actual data before `vle` is freed.  
Any hardcoded type not present in the data is flagged and removed.

In [4]:
print('Validating Task 2 handoff activity_type coverage ...')

actual_types = set().union(*(
    set(df['activity_type'].dropna().astype(str).unique())
    for df in student_vle_by_week.values()
))
missing_from_data = [a for a in VLE_ACT_TYPES if a not in actual_types]
if missing_from_data:
    print(f'WARNING: Not found in handoff data, removing from VLE_ACT_TYPES: {missing_from_data}')
    VLE_ACT_TYPES[:] = [a for a in VLE_ACT_TYPES if a in actual_types]

for name, df in student_vle_by_week.items():
    n_unmatched = int(df['activity_type'].isna().sum())
    pct = n_unmatched / len(df) * 100
    max_date = int(df['date'].max())
    cutoff = CUTOFFS[name]
    status = 'OK' if max_date <= cutoff and n_unmatched == 0 else 'CHECK'
    print(f'  [{status}] {name}: max date={max_date}, unmatched activity_type={n_unmatched} ({pct:.2f}%)')

print(f'Validated VLE_ACT_TYPES: {VLE_ACT_TYPES}')


Validating Task 2 handoff activity_type coverage ...
  [OK] week2: max date=13, unmatched activity_type=0 (0.00%)
  [OK] week4: max date=27, unmatched activity_type=0 (0.00%)
  [OK] week6: max date=41, unmatched activity_type=0 (0.00%)
  [OK] week8: max date=55, unmatched activity_type=0 (0.00%)
Validated VLE_ACT_TYPES: ['oucontent', 'quiz', 'resource', 'homepage', 'subpage', 'glossary', 'oucollaborate', 'forumng']


## 5. Fit imputers and report missingness
Fit once — only `.transform()` called inside the build loop.  
**Limitation note:** `studentRegistration` (`date_unregistration`) is not used. Students who
withdrew before the cutoff date will appear as 0-click rows, not missing rows. This is a known
limitation — document in Table 2.3 Notes for VLE features.

In [5]:
_si_for_fit = student_info[CAT_COLS + NUM_COLS].copy()
_si_for_fit[CAT_COLS] = _si_for_fit[CAT_COLS].apply(
    lambda c: c.str.strip() if c.dtype == object else c
)
_si_for_fit[CAT_COLS] = _si_for_fit[CAT_COLS].replace({'': np.nan, '?': np.nan})

print('Missingness in demographic columns (same across all 4 cutoffs):')
missing = _si_for_fit[CAT_COLS + NUM_COLS].isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    for col, n in missing.items():
        print(f'  {col}: {n} ({n/len(student_info)*100:.1f}%)')
else:
    print('  None')

del _si_for_fit
print('Data cleaned. Imputation is strictly deferred to downstream Task 4/5 pipelines to prevent leakage.')

Missingness in demographic columns (same across all 4 cutoffs):
  imd_band: 1111 (3.4%)
Data cleaned. Imputation is strictly deferred to downstream Task 4/5 pipelines to prevent leakage.


## 6. Aggregation and build functions
**Leakage rule:** only `date <= cutoff` interactions used.  
**VLE features per cutoff:**
- `total_clicks` — sum of all clicks up to cutoff
- `active_days` — distinct days with ≥1 click (negative dates = pre-course access, included)
- Per-activity-type: `{type}_clicks` for each type in `VLE_ACT_TYPES` (validated in cell 4)

Note: `CAT_COLS`, `NUM_COLS`, `VLE_ACT_TYPES` are module-level constants used inside these functions.

In [6]:
def aggregate_vle(df_vle: pd.DataFrame, cutoff: int, suffix: str) -> pd.DataFrame:
    """
    Aggregate VLE interactions up to `cutoff` day (inclusive).
    Produces total_clicks, active_days, and one column per type in VLE_ACT_TYPES.
    """
    group_keys = ['id_student', 'code_module', 'code_presentation']
    filtered = df_vle[df_vle['date'] <= cutoff]

    total_clicks = (
        filtered.groupby(group_keys)['sum_click'].sum()
        .rename(f'total_clicks_{suffix}')
    )
    active_days = (
        filtered.groupby(group_keys)['date'].nunique()
        .rename(f'active_days_{suffix}')
    )
    act_series = []
    for act in VLE_ACT_TYPES:
        s = (
            filtered[filtered['activity_type'] == act]
            .groupby(group_keys)['sum_click'].sum()
            .rename(f'{act}_clicks_{suffix}')
            .reindex(total_clicks.index, fill_value=0)
        )
        act_series.append(s)

    return pd.concat([total_clicks, active_days] + act_series, axis=1).reset_index()


def build_dataset(demographics: pd.DataFrame,
                  student_vle: pd.DataFrame,
                  cutoff: int,
                  suffix: str) -> pd.DataFrame:
    """Build a leakage-safe feature table for one week cutoff."""
    act_cols   = [f'{a}_clicks_{suffix}' for a in VLE_ACT_TYPES]
    click_cols = [f'total_clicks_{suffix}', f'active_days_{suffix}'] + act_cols

    vle_agg = aggregate_vle(student_vle, cutoff, suffix)
    df = demographics.merge(
        vle_agg, on=['id_student', 'code_module', 'code_presentation'], how='left'
    )
    
    # Fill VLE missingness (0 clicks means 0 activity)
    df[click_cols] = df[click_cols].fillna(0)

    # Clean demographics but LEAVE structural NaNs intact for ML pipelines
    df[CAT_COLS] = df[CAT_COLS].apply(lambda c: c.str.strip() if c.dtype == object else c)
    df[CAT_COLS] = df[CAT_COLS].replace({'': np.nan, '?': np.nan})
    
    # Use pandas nullable integer type ('Int64') so columns stay int even with NaNs
    df['num_of_prev_attempts'] = df['num_of_prev_attempts'].astype('Int64')
    df['studied_credits']      = df['studied_credits'].astype('Int64')

    ordered = (
        ['id_student', 'code_module', 'code_presentation']
        + CAT_COLS + NUM_COLS + click_cols + ['label']
    )
    return df[ordered]

print('Functions defined.')

Functions defined.


## 7. Build and save all 4 datasets

In [7]:
datasets = {}
for name, cutoff in CUTOFFS.items():
    student_vle_cutoff = student_vle_by_week[name]
    print(f'\nBuilding {name} from Task 2 handoff (date <= {cutoff}) ...')
    df_out = build_dataset(
        student_info, student_vle_cutoff, cutoff, name
    )
    datasets[name] = df_out
    out_path = OUT_DIR / f'{name}_features.csv'
    df_out.to_csv(out_path, index=False)
    print(f'  Shape: {df_out.shape}  |  Missing (Deferred): {df_out.isnull().sum().sum()}  |  Saved -> {out_path}')
    print(f'  Label dist: {df_out["label"].value_counts().to_dict()}')



Building week2 from Task 2 handoff (date <= 13) ...


/tmp/ipykernel_7484/3437097753.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['sum_click'].sum()
/tmp/ipykernel_7484/3437097753.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['date'].nunique()
/tmp/ipykernel_7484/3437097753.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_keys)['sum_click'].sum()


  Shape: (32593, 22)  |  Missing (Deferred): 1111  |  Saved -> /home/njabulo/Documents/Wits/Final year/Semester_1/Machine_Learning/2026/Projects/Group/ELEN4025_Group_Assignment/data/processed/week2_features.csv
  Label dist: {0: 17208, 1: 15385}

Building week4 from Task 2 handoff (date <= 27) ...


/tmp/ipykernel_7484/3437097753.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['sum_click'].sum()
/tmp/ipykernel_7484/3437097753.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['date'].nunique()
/tmp/ipykernel_7484/3437097753.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_keys)['sum_click'].sum()


  Shape: (32593, 22)  |  Missing (Deferred): 1111  |  Saved -> /home/njabulo/Documents/Wits/Final year/Semester_1/Machine_Learning/2026/Projects/Group/ELEN4025_Group_Assignment/data/processed/week4_features.csv
  Label dist: {0: 17208, 1: 15385}

Building week6 from Task 2 handoff (date <= 41) ...


/tmp/ipykernel_7484/3437097753.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['sum_click'].sum()
/tmp/ipykernel_7484/3437097753.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['date'].nunique()
/tmp/ipykernel_7484/3437097753.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_keys)['sum_click'].sum()


  Shape: (32593, 22)  |  Missing (Deferred): 1111  |  Saved -> /home/njabulo/Documents/Wits/Final year/Semester_1/Machine_Learning/2026/Projects/Group/ELEN4025_Group_Assignment/data/processed/week6_features.csv
  Label dist: {0: 17208, 1: 15385}

Building week8 from Task 2 handoff (date <= 55) ...


/tmp/ipykernel_7484/3437097753.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['sum_click'].sum()
/tmp/ipykernel_7484/3437097753.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  filtered.groupby(group_keys)['date'].nunique()
/tmp/ipykernel_7484/3437097753.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(group_keys)['sum_click'].sum()


  Shape: (32593, 22)  |  Missing (Deferred): 1111  |  Saved -> /home/njabulo/Documents/Wits/Final year/Semester_1/Machine_Learning/2026/Projects/Group/ELEN4025_Group_Assignment/data/processed/week8_features.csv
  Label dist: {0: 17208, 1: 15385}


## 8. Sanity checks

In [8]:
print('=' * 60)
print('SANITY CHECKS')
print('=' * 60)

all_passed   = True
baseline_len = len(datasets['week2'])

for name, df in datasets.items():
    suffix     = name
    cutoff_val = CUTOFFS[name]
    act_cols   = [f'{a}_clicks_{suffix}' for a in VLE_ACT_TYPES]
    click_cols = [f'total_clicks_{suffix}', f'active_days_{suffix}'] + act_cols
    errors = []

    # Only check VLE columns for missing values; demographics are intentionally left for the pipeline
    vle_missing = df[click_cols].isnull().sum().sum()
    if vle_missing > 0:
        errors.append(f'{vle_missing} missing values in VLE features')
        
    if set(df['label'].unique()) != {0, 1}:
        errors.append(f'Unexpected labels: {set(df["label"].unique())}')
    for col in click_cols:
        if (df[col] < 0).any():
            errors.append(f'{col} has negative values')

    n_dupes = df.duplicated(subset=['id_student','code_module','code_presentation']).sum()
    if n_dupes > 0:
        errors.append(f'{n_dupes} duplicate rows')

    # Verify join correctness: each activity type sum <= total (subset by definition)
    for act_col in act_cols:
        bad = (df[act_col] > df[f'total_clicks_{suffix}']).sum()
        if bad > 0:
            errors.append(f'Join error: {bad} rows where {act_col} > total_clicks')

    max_possible_days = cutoff_val - EARLIEST_DATE + 1
    n_impossible = (df[f'active_days_{suffix}'] > max_possible_days).sum()
    if n_impossible > 0:
        errors.append(f'{n_impossible} rows active_days > {max_possible_days}')
    if len(df) != baseline_len:
        errors.append(f'Row count {len(df)} differs from week2 baseline {baseline_len}')
        
    for col in ['num_of_prev_attempts', 'studied_credits']:
        if not pd.api.types.is_integer_dtype(df[col]):
            errors.append(f'{col} is not an integer type')

    status = 'PASS' if not errors else 'FAIL'
    if errors: all_passed = False
    print(f'[{status}] {name}: {", ".join(errors) if errors else "all checks passed"}')

print('\nMonotonicity checks …')
for w_early, w_late in zip(['week2','week4','week6'], ['week4','week6','week8']):
    keys = ['id_student', 'code_module', 'code_presentation']
    merged = datasets[w_early][keys + [f'total_clicks_{w_early}']].merge(
        datasets[w_late][keys + [f'total_clicks_{w_late}']], on=keys
    )
    v = (merged[f'total_clicks_{w_late}'] < merged[f'total_clicks_{w_early}']).sum()
    status = 'FAIL' if v > 0 else 'PASS'
    if v > 0: all_passed = False
    print(f'[{status}] Monotonicity {w_early} → {w_late}' + (f': {v} violations' if v > 0 else ''))

print()
print('All datasets clean — ready for handoff.' if all_passed else 'Checks FAILED — fix before committing.')

SANITY CHECKS
[PASS] week2: all checks passed
[PASS] week4: all checks passed
[PASS] week6: all checks passed
[PASS] week8: all checks passed

Monotonicity checks …
[PASS] Monotonicity week2 → week4
[PASS] Monotonicity week4 → week6
[PASS] Monotonicity week6 → week8

All datasets clean — ready for handoff.


## 9. Generate Table 2.3 (Features Table for PDF submission)
Produces `features_table.csv` to directly fill Table 2.3 in the brief.  
Week 4 used as the representative sample for counts — demographics are identical across all cutoffs.

In [9]:
ref_df     = datasets['week4']
ref_suffix = 'week4'
feature_records = []

# ── Demographic features ────────────────────────────────────────
demo_meta = [
    ('gender',               'studentInfo.csv', 'gender',               'cat', 'Student gender',                       'direct', 'None'),
    ('region',               'studentInfo.csv', 'region',               'cat', 'UK region of student',                 'direct', 'None'),
    ('highest_education',    'studentInfo.csv', 'highest_education',    'cat', 'Highest prior qualification',          'direct', 'None'),
    ('imd_band',             'studentInfo.csv', 'imd_band',             'cat', 'Index of multiple deprivation decile', 'direct', 'None'),
    ('age_band',             'studentInfo.csv', 'age_band',             'cat', 'Age band at registration',             'direct', 'None'),
    ('disability',           'studentInfo.csv', 'disability',           'cat', 'Declared disability status',           'direct', 'None'),
    ('num_of_prev_attempts', 'studentInfo.csv', 'num_of_prev_attempts', 'num', 'Number of prior module attempts',      'direct', 'None'),
    ('studied_credits',      'studentInfo.csv', 'studied_credits',      'num', 'Total credits registered for',         'direct', 'None'),
]

for fname, src, orig, dtype, defn, how, leak in demo_meta:
    col = ref_df[fname]
    feature_records.append({
        'Feature': fname, 'Source CSV(s)': src, 'Original column(s)': orig,
        'Data type': dtype, 'Definition': defn, 'How computed': how,
        'Week availability': '2/4/6/8',
        'Missing count': int(col.isnull().sum()),
        'Outliers count': count_outliers_iqr(col) if dtype == 'num' else 'N/A',
        # Duplicates are a row-level property — 0 confirmed by sanity checks
        'Duplicate count': '0 (row-level, verified by sanity checks)',
        'Leakage risk': leak,
        'Notes': 'Imputation deferred to modeling pipeline (structural NaN only)',
    })

# ── VLE features ────────────────────────────────────────────────
vle_limitation = ('Limitation: students who withdrew before cutoff appear as 0-click '
                   'rows, not missing. studentRegistration not used.')
vle_meta = (
    [('total_clicks', 'studentVle.csv', 'sum_click', 'Total VLE clicks up to cutoff', 'sum(sum_click) where date<=cutoff'),
     ('active_days',  'studentVle.csv', 'date',      'Distinct days with ≥1 click',   'nunique(date) where date<=cutoff')]
    + [(f'{a}_clicks', 'studentVle.csv + vle.csv', 'sum_click, activity_type',
        f'Clicks on {a} activity up to cutoff', f'sum where activity_type={a}')
       for a in VLE_ACT_TYPES]
)

for fname, src, orig, defn, how in vle_meta:
    col_name = f'{fname}_{ref_suffix}'
    col = ref_df.get(col_name, pd.Series(dtype=float))
    feature_records.append({
        'Feature': fname, 'Source CSV(s)': src, 'Original column(s)': orig,
        'Data type': 'num', 'Definition': defn, 'How computed': how,
        'Week availability': '2/4/6/8',
        'Missing count': int(col.isnull().sum()) if len(col) > 0 else 0,
        'Outliers count': count_outliers_iqr(col) if len(col) > 0 else 0,
        'Duplicate count': 'N/A',
        'Leakage risk': 'None',
        'Notes': 'No-activity students get 0. ' + vle_limitation,
    })

features_table = pd.DataFrame(feature_records)
ft_path = os.path.join(OUT_DIR, 'features_table.csv')
features_table.to_csv(ft_path, index=False)
print(f'Features table → {ft_path}  ({len(features_table)} features)')
print(features_table[['Feature','Data type','Missing count','Outliers count','Leakage risk']].to_string(index=False))

Features table → /home/njabulo/Documents/Wits/Final year/Semester_1/Machine_Learning/2026/Projects/Group/ELEN4025_Group_Assignment/data/processed/features_table.csv  (18 features)
             Feature Data type  Missing count Outliers count Leakage risk
              gender       cat              0            N/A         None
              region       cat              0            N/A         None
   highest_education       cat              0            N/A         None
            imd_band       cat           1111            N/A         None
            age_band       cat              0            N/A         None
          disability       cat              0            N/A         None
num_of_prev_attempts       num              0           4172         None
     studied_credits       num              0            350         None
        total_clicks       num              0           1991         None
         active_days       num              0            366         None
    ou

## 10. Summary table

In [10]:
summary_rows = []
for name, df in datasets.items():
    suffix  = name
    n_fav   = (df['label'] == 1).sum()
    n_unfav = (df['label'] == 0).sum()
    n_feats = len([c for c in df.columns
                   if c not in ['id_student','code_module','code_presentation','label']])
    summary_rows.append({
        'Dataset'            : name,
        'Cutoff (day)'       : CUTOFFS[name],
        'Rows'               : len(df),
        'Features'           : n_feats,
        'Favourable (1)'     : n_fav,
        'Unfavourable (0)'   : n_unfav,
        'Imbalance %'        : round(n_unfav / len(df) * 100, 1),
        'Median total clicks': round(df[f'total_clicks_{suffix}'].median(), 1),
    })
print(pd.DataFrame(summary_rows).to_string(index=False))

Dataset  Cutoff (day)  Rows  Features  Favourable (1)  Unfavourable (0)  Imbalance %  Median total clicks
  week2            13 32593        18           15385             17208         52.8                 92.0
  week4            27 32593        18           15385             17208         52.8                176.0
  week6            41 32593        18           15385             17208         52.8                224.0
  week8            55 32593        18           15385             17208         52.8                271.0
